# Controlling the Thread Pool in `feos`

Several functions in `feos` use [Rayon](https://github.com/rayon-rs/rayon) for parallelism.
By default, Rayon uses all logical CPUs available on your machine, which is usually what you want when working on your local machine. 
In other environments, for example HPC clusters, you may want to limit the number of threads to match your job allocation.

There are three ways to configure this, in order of priority:

- `FEOS_MAX_THREADS` environment variable: for HPC or "script" environments, defined before launching Python
- `feos.set_num_threads()`:  for interactive use, at the top of a script or notebook
- Do nothing: local machines where using all cores is fine

You can get the number of threads configured via `feos.get_num_threads()`.

## Important
- The thread pool can only be configured **once** per Python process.
- Whichever method runs first wins. Any later attempt to change it will have no effect and a warning will be emitted.
- Calling `get_num_threads` without setting `FEOS_MAX_THREADS` or `set_num_threads` will initialze the thread pool with the default (all logical CPUs).
- To test the different behaviour in this notebook, restart the kernel and start from the respective cell you want to test.

## Method 1: Environment variable (recommended for HPC)

Set `FEOS_MAX_THREADS` **before** starting Python or launching your Jupyter kernel.
The thread pool is initialized automatically when `feos` is imported.

In a Slurm job script:

```bash
#!/bin/bash
#SBATCH --cpus-per-task=8

export FEOS_MAX_THREADS=$SLURM_CPUS_PER_TASK
python my_script.py
```

Or in a terminal before starting Jupyter:

```bash
export FEOS_MAX_THREADS=4
jupyter lab
```

You can verify it was picked up after importing:

In [3]:
import feos

# If FEOS_MAX_THREADS was set before starting Python, the pool
# was already configured at import time.
print(f"Active threads: {feos.get_num_threads()}")

Active threads: 10


## Method 2: `set_num_threads()` (interactive use)

If you did not set the environment variable, you can configure the thread pool
programmatically. This must be done **before any parallel computation is triggered**.

In a notebook or script, call it immediately after importing `feos`: (if the code below emits a warning, restart the kernel and run the code below as first cell)

In [1]:
import feos

feos.set_num_threads(4)

print(f"Active threads: {feos.get_num_threads()}")

Active threads: 4


You can also read the thread count from `SLURM_CPUS_PER_TASK` manually if you prefer
to keep configuration in Python rather than in your shell environment:

In [2]:
import os
import feos

n_threads = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))
feos.set_num_threads(n_threads)

print(f"Active threads: {feos.get_num_threads()}")

Active threads: 4


/var/folders/3s/t93ws1md04qdbbq5d1jdz8640000gn/T/ipykernel_9962/4108200440.py:5: UserWarning: set_num_threads(10) without effect: The thread pool was already initialized with 4 thread(s) Call set_num_threads() before any parallel work or set FEOS_MAX_THREADS before starting Python.
  feos.set_num_threads(n_threads)


## Method 3: Do nothing (Rayon default)

If you neither set `FEOS_MAX_THREADS` nor call `set_num_threads()`, Rayon will initialize the thread pool lazily the first time a parallel function is called, using all available logical CPUs. This is usually the right choice on a local workstation.
Note that calling `get_num_threads` will set the threads to the default.

In [ ]:
import feos

# No configuration — Rayon will use all logical CPUs.
# get_num_threads() triggers lazy initialization if not already done.
print(f"Active threads: {feos.get_num_threads()}")

## What happens if you call `set_num_threads()` too late?

If the pool is already initialized — because `FEOS_MAX_THREADS` was set, or because
a parallel function (or `get_num_threads()`) has already run — `set_num_threads()`
has no effect and emits a warning:

In [ ]:
import feos

print(feos.get_num_threads())  # triggers lazy initialization

feos.set_num_threads(2)        # UserWarning: had no effect

## Summary

```
                        import feos
                             │
               FEOS_MAX_THREADS set?
              ┌──────┴───────┐
             Yes             No
              │              │
    Pool initialized    set_num_threads() called?
    with env var value  ┌──────┴───────┐
                       Yes             No
                        │              │
             Pool initialized    Pool initialized lazily
             with given value    on first parallel call
                                 (all logical CPUs)
```

## Example

The following example calculates vapor pressures and derivatives w.r.t. the model's parameters in parallel.
Set the number of threads to check the impact on calculation time (restart kernel as before to change the threads).

In [1]:
import feos
import numpy as np
import timeit

# modify the number of threads and rerun the cell below to see impact
feos.set_num_threads(0) 

n = 1_000_000
fit_params = ["m", "sigma", "epsilon_k"]

# order: m, sigma, epsilon_k, mu
parameters = np.array([[1.5, 3.4, 230.0, 2.3]] * n)
temperature = np.expand_dims(np.linspace(250.0, 400.0, n), 1)
eos = feos.EquationOfStateAD.PcSaftNonAssoc

In [5]:
n_runs = 5
n_threads = feos.get_num_threads()
elapsed = np.mean(timeit.repeat(
    lambda: feos.vapor_pressure_derivatives(eos, fit_params, parameters, temperature),
    number=1,
    repeat=n_runs
))

print(f"Mean of {n_runs} runs ({n} VLEs each): {elapsed:.3f}s using {n_threads} thread(s)")

Mean of 5 runs (1000000 VLEs each): 0.559s using 10 thread(s)
